In [0]:
 %run /Configurations/Init_Scripts

In [0]:
def mergeIntoRulesEngineAudit(df_RulesCalculation, RuleCategory = None,audit_table = 'promotion.LOG_RulesEngineCalculation'):
    try:
        df_RulesCalculation = df_RulesCalculation.withColumn('RULECATEGORY',lit(RuleCategory))
        
        # Convert all columns to string and uppercase, and handle duplicates by adding suffix
        columns = df_RulesCalculation.columns
        new_columns = []
        for c in columns:
            new_col = c.upper()
            if new_col in new_columns:
                suffix = 1
                while f"{new_col}_{suffix}" in new_columns:
                    suffix += 1
                new_col = f"{new_col}_{suffix}"
            new_columns.append(new_col)

        df_RulesCalculation = df_RulesCalculation.select([upper(col(c).cast("string")).alias(new_col) for c, new_col in zip(columns, new_columns)])

        df_RulesCalculation = (df_RulesCalculation.withColumn('CREATEDBY',lit('ExceptionInsert'))
                                        .withColumn('CREATEDDATE',lit(current_timestamp())))
        
        df_RulesCalculation.write.mode('append').option('mergeSchema', 'true').saveAsTable(audit_table)
    except Exception as e:
        print(str(e))

In [0]:
def logValidateP3Flag(df_CycleData,Cycletype=None):
    df_DIMSmartCard = spark.read.table('Promotion.DIM_SmartCard')
    df_DIMPromotion = spark.read.table('Promotion.DIM_Promotion')
    if Cycletype == 'CyclesOverride':
        df_DIMAccount = spark.read.table('Promotion.DIM_Account')
        df_CycleData_AGG = (df_CycleData
                        .join(broadcast(df_DIMPromotion),'PromotionUUID','inner')
                        .join(df_DIMAccount, ['ShipToAccountID','PromotionUUID'], 'inner')
                        .join(df_DIMSmartCard, ['SmartCardSerialNumber','PromotionUUID'], 'left')
                        .withColumn('P3EligibilityFlag',when(
                            (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMPromotion.EffectiveDate, df_DIMPromotion.TerminationDate))
                            &
                            ((df_CycleData.FileNameDeviceTypeCd == df_DIMPromotion.DeviceTypeCd) | (df_CycleData.DeviceTypeOverrideFlag == True))
                            &
                            ((df_DIMAccount.PerPatientPricingFlag == 1) | (df_CycleData.AccountP3FlagOverrideFlag == True))
                            &
                            ((df_CycleData.HdrStartDateGeneratedDt >= df_DIMAccount.EffectiveDate) | (df_CycleData.AccountEffectiveDateOverrideFlag == True))
                            &
                            ((df_CycleData.HdrStartDateGeneratedDt <= df_DIMAccount.TerminationDate) | (df_CycleData.AccountTerminationDateOverrideFlag == True))
                            &
                            ((df_DIMSmartCard.SmartCardSerialNumber.isNotNull()) | (df_CycleData.SmartCardOverrideFlag == True))
                            &
                            ((df_CycleData.HdrStartDateGeneratedDt >= df_DIMSmartCard.EffectiveDate) | (df_CycleData.SmartCardEffectiveDateOverrideFlag == True) | (df_CycleData.SmartCardOverrideFlag == True))
                            &
                            ((df_CycleData.HdrStartDateGeneratedDt <= df_DIMSmartCard.TerminationDate) | (df_CycleData.SmartCardTerminationDateOverrideFlag == True) | (df_CycleData.SmartCardOverrideFlag == True)),True
                            ).otherwise(lit(False)))
                        .withColumnRenamed('HdrStartDateGeneratedDt','CycleDate') 
                        .withColumn('CycleMonth',date_format(col('CycleDate'), 'yyyy-MM'))      
                        .select('CycleID', 'CycleDate', 'P3EligibilityFlag','CycleMonth')                
                        .withColumn('CreatedDate', lit(current_timestamp()))
                        )
    else:
        df_DIMAccount = spark.read.table('Promotion.DIM_Account').filter("PerPatientPricingFlag = 1")
        
        df_CycleData_AGG = (df_CycleData
                .join(df_DIMAccount, ['ShipToAccountID'], 'left')
                .join(broadcast(df_DIMPromotion),
                    (df_CycleData.FileNameDeviceTypeCd == df_DIMPromotion.DeviceTypeCd) & 
                    (df_DIMAccount.PromotionUUID == df_DIMPromotion.PromotionUUID),
                    'left')
                .join(df_DIMSmartCard, ['SmartCardSerialNumber','PromotionUUID'], 'left')
                .withColumn('P3EligibilityFlag',
                    when(
                        (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMPromotion.EffectiveDate, df_DIMPromotion.TerminationDate)) &
                        (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMAccount.EffectiveDate, df_DIMAccount.TerminationDate)) &
                        (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMSmartCard.EffectiveDate, df_DIMSmartCard.TerminationDate)) &
                        (df_DIMPromotion.PromotionUUID.isNotNull()) &
                        (col('ShipToAccountUUID').isNotNull()) &
                        (col('SmartCardSerialNumber').isNotNull())
                        , lit(True)
                    ).otherwise(lit(False)))
                        .withColumnRenamed('HdrStartDateGeneratedDt','CycleDate') 
                        .withColumn('CycleMonth',date_format(col('CycleDate'), 'yyyy-MM'))       
                        .select('CycleID', 'CycleDate', 'P3EligibilityFlag','CycleMonth')                
                        .withColumn('CreatedDate', lit(current_timestamp()))                    
                        )
        
    df_CycleData_AGG.write.mode('append').option('mergeSchema', 'true').saveAsTable('promotion.log_CyclesValidation')

In [0]:
def setWorkFlowStatus_CyclesOverride (df_DIMCyclesOverride,WorkFlowStatus,UpdatedBy):
    df_DIMCyclesOverride = (df_DIMCyclesOverride
                            .withColumn('WorkFlowStatus',lit(WorkFlowStatus))
                            .withColumn('UpdatedBy',lit(UpdatedBy))
                            .withColumn('UpdatedDate',lit(current_timestamp()))
                            )

    dl_DIMCyclesOverride = DeltaTable.forName(spark, 'promotion.DIM_CyclesOverride')
    (dl_DIMCyclesOverride.alias("tgt")
            .merge(df_DIMCyclesOverride.alias("src"),
                ("tgt.CycleId = src.CycleId AND tgt.PromotionUUID = src.PromotionUUID AND tgt.CreatedDate = src.CreatedDate "))
            .whenMatchedUpdate(set ={
                "tgt.WorkflowStatus": "src.WorkFlowStatus",
                "tgt.UpdatedBy": "src.UpdatedBy",
                "tgt.UpdatedDate": "src.UpdatedDate",                                
                })
    .execute())


In [0]:
def applyRules(input_df,RuleCategory):
        rules = (spark.read.table('promotion.DIM_RuleEngineConfig')
                .filter((col('Active') == True )
                        & 
                        (current_date().between(col('effectiveDateTime'),col('terminationDateTime')))
                        & 
                        (col('RuleCategory') == RuleCategory))
                .orderBy('RuleOrder')
                )

        promotions = (spark.read.table('promotion.DIM_Promotion')
                        .filter((col('Active') == True )
                                & 
                                (current_date().between(col('effectiveDate'),col('terminationDate'))))
                        .select('PromotionName','PromotionUUID')
                        .orderBy('PromotionName')
                        )

        output_df = None

        for row in promotions.collect():
                promotionUUID = row.PromotionUUID
                promotionName = row.PromotionName
                
                promotion_df = input_df.filter(col('PromotionUUID') == lit(promotionUUID))
        
                for rule_row in rules.collect():
                        ruleName = rule_row.RuleName
                        ruleCode = rule_row.RuleCode
                        rule_promotionUUID = rule_row.PromotionUUID
                        if rule_promotionUUID == promotionUUID:
                            promotion_df = promotion_df.withColumn(ruleName, expr(ruleCode))

                if output_df is None:
                        output_df = promotion_df
                else:
                        output_df = output_df.unionByName(promotion_df,True)
        mergeIntoRulesEngineAudit(output_df,RuleCategory)
        return output_df


In [0]:
def getValidPromotionCycles(df_CycleData):
    df_DIMAccount = spark.read.table('Promotion.DIM_Account').filter("PerPatientPricingFlag = 1")
    df_DIMSmartCard = spark.read.table('Promotion.DIM_SmartCard')
    df_DIMPromotion = spark.read.table('Promotion.DIM_Promotion')
    df_ApplicatorConfig = spark.read.table('Silverzone.DIMApplicatorVerCd').select(col('SSAPPIdentID').alias('ApplicatorConfigurantionNbr'), col('FullEfficacy'))

    logValidateP3Flag(df_CycleData)    

    df_CycleData_AGG = (df_CycleData
                .join(broadcast(df_DIMPromotion),df_CycleData.FileNameDeviceTypeCd == df_DIMPromotion.DeviceTypeCd,'inner')
                .join(df_DIMAccount, ['ShipToAccountID','PromotionUUID'], 'inner')
                .join(df_DIMSmartCard, ['SmartCardSerialNumber','PromotionUUID'], 'inner')
                .join(broadcast(df_ApplicatorConfig), ['ApplicatorConfigurantionNbr'], 'left')
                .withColumn('IsValidEfficacy',
                        when(
                            when(
                                col('BeginTimeStamp') > col('EndTimeStamp'),
                                expr("timestampdiff(MINUTE, EndTimeStamp, BeginTimeStamp)")
                            ).otherwise(
                                expr("timestampdiff(MINUTE, BeginTimeStamp, EndTimeStamp)")
                            ) >= col('FullEfficacy'),
                            lit('Yes')
                        ).otherwise(lit('No')))
                .filter(
                        (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMPromotion.EffectiveDate, df_DIMPromotion.TerminationDate)) &
                        (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMAccount.EffectiveDate, df_DIMAccount.TerminationDate)) &
                        (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMSmartCard.EffectiveDate, df_DIMSmartCard.TerminationDate)))
                .withColumnRenamed('HdrStartDateGeneratedDt','CycleDate')
                    )

    df_CycleData_AGG_Valid = (df_CycleData_AGG
                                .groupBy('CoolSculptingID','SoldToAccountID','ShipToAccountID', 'ShipToAccountUUID','PromotionUUID', 'CycleDate','PromotionRuleType','PlanType','IsValidEfficacy')
                                .agg(count('*').alias('CycleCount'), 
                                    min('BeginTimeStamp').alias('MinCycleTmstmp'),
                                    collect_set('SmartCardSerialNumber').alias('SmartCardSerialNumberList'),
                                    collect_set('ApplicatorSerialNbr').alias('ApplicatorSerialNbrList'),
                                    collect_set('CycleID').alias('CycleIDList')
                                   )
                                .withColumn('CycleDataUUID',expr('uuid()'))
                                .withColumn('SmartCardSerialNumber', expr("concat_ws(', ', SmartCardSerialNumberList)"))
                                .withColumn('ApplicatorSerialNumber', expr("concat_ws(', ', ApplicatorSerialNbrList)"))
                                .withColumn('CycleID', expr("concat_ws(', ', CycleIDList)"))               
                                .drop('SmartCardSerialNumberList','ApplicatorSerialNbrList','CycleIDList')
                                )
    return df_CycleData_AGG_Valid

In [0]:
def getValidPromotionCycles_Overrides(df_CycleData):

    df_DIMAccount = spark.read.table('Promotion.DIM_Account')
    df_DIMSmartCard = spark.read.table('Promotion.DIM_SmartCard')
    df_DIMPromotion = spark.read.table('Promotion.DIM_Promotion')
    df_ApplicatorConfig = spark.read.table('Silverzone.DIMApplicatorVerCd').select(col('SSAPPIdentID').alias('ApplicatorConfigurantionNbr'), col('FullEfficacy'))
    
    df_UlLogs = (spark.read.table('silverzone.factullogs')
                            .withColumnRenamed('CardSPSerialNbr', 'SmartCardSerialNumber')
                            .withColumnRenamed('HdrStartTimeGeneratedTmstmp', 'BeginTimeStamp')
                            .withColumnRenamed('EventEndTmstmp', 'EndTimeStamp')
                            .withColumn('PlanType', coalesce(col('PlanType'), lit('ComprehensiveTreatmentPlan')))
                            .withColumn('ApplicatorConfigurantionNbr', left(col('ApplicatorConfigurantionNbr'), lit(7)))             
                            .select('CycleID', 'ShipToAccountID', 'SoldToAccountID', 'HdrStartDateGeneratedDt', 'BeginTimeStamp', 'EndTimeStamp','SmartCardSerialNumber', 'CoolSculptingID', 'FileNameDeviceTypeCd', 'PlanType','ApplicatorConfigurantionNbr','ApplicatorSerialNbr')
                            )

    df_CycleData = df_UlLogs.join(df_CycleData,'CycleID','inner')
    logValidateP3Flag(df_CycleData,'CyclesOverride')
    df_CycleData_AGG = (df_CycleData
                    .join(broadcast(df_DIMPromotion),'PromotionUUID','inner')
                    .join(df_DIMAccount, ['ShipToAccountID','PromotionUUID'], 'inner')
                    .join(df_DIMSmartCard, ['SmartCardSerialNumber','PromotionUUID'], 'left')
                    .join(df_ApplicatorConfig, ['ApplicatorConfigurantionNbr'], 'left')
                    .withColumn('IsValidEfficacy',
                        when(
                            when(
                                col('BeginTimeStamp') > col('EndTimeStamp'),
                                expr("timestampdiff(MINUTE, EndTimeStamp, BeginTimeStamp)")
                            ).otherwise(
                                expr("timestampdiff(MINUTE, BeginTimeStamp, EndTimeStamp)")
                            ) >= col('FullEfficacy'),
                            lit('Yes')
                        ).otherwise(lit('No'))
                    )
                    .where(
                        (df_CycleData.HdrStartDateGeneratedDt.between(df_DIMPromotion.EffectiveDate, df_DIMPromotion.TerminationDate))
                        &
                        ((df_CycleData.FileNameDeviceTypeCd == df_DIMPromotion.DeviceTypeCd) | (df_CycleData.DeviceTypeOverrideFlag == True))
                        &
                        ((df_DIMAccount.PerPatientPricingFlag == 1) | (df_CycleData.AccountP3FlagOverrideFlag == True))
                        &
                        ((df_CycleData.HdrStartDateGeneratedDt >= df_DIMAccount.EffectiveDate) | (df_CycleData.AccountEffectiveDateOverrideFlag == True))
                        &
                        ((df_CycleData.HdrStartDateGeneratedDt <= df_DIMAccount.TerminationDate) | (df_CycleData.AccountTerminationDateOverrideFlag == True))
                        &
                        ((df_DIMSmartCard.SmartCardSerialNumber.isNotNull()) | (df_CycleData.SmartCardOverrideFlag == True))
                        &
                        ((df_CycleData.HdrStartDateGeneratedDt >= df_DIMSmartCard.EffectiveDate) | (df_CycleData.SmartCardEffectiveDateOverrideFlag == True) | (df_CycleData.SmartCardOverrideFlag == True))
                        &
                        ((df_CycleData.HdrStartDateGeneratedDt <= df_DIMSmartCard.TerminationDate) | (df_CycleData.SmartCardTerminationDateOverrideFlag == True) | (df_CycleData.SmartCardOverrideFlag == True))
                        )
                    .groupBy('CoolSculptingID','SoldToAccountID','ShipToAccountID', 'ShipToAccountUUID','PromotionUUID', 'HdrStartDateGeneratedDt','PromotionRuleType','PlanType','IsValidEfficacy')
                    .agg(count('*').alias('CycleCount'), 
                         min('BeginTimeStamp').alias('MinCycleTmstmp'),
                         collect_set('SmartCardSerialNumber').alias('SmartCardSerialNumberList'),
                         collect_set('ApplicatorSerialNbr').alias('ApplicatorSerialNbrList'),
                         collect_set('CycleID').alias('CycleIDList')
                         )
                    .withColumnRenamed('HdrStartDateGeneratedDt','CycleDate')
                    .withColumn('CycleDataUUID',expr('uuid()'))
                    .withColumn('SmartCardSerialNumber', expr("concat_ws(', ', SmartCardSerialNumberList)"))
                    .withColumn('ApplicatorSerialNumber', expr("concat_ws(', ', ApplicatorSerialNbrList)"))
                    .withColumn('CycleID', expr("concat_ws(', ', CycleIDList)"))
                    .drop('SmartCardSerialNumberList','ApplicatorSerialNbrList','CycleIDList')
                )
    return df_CycleData_AGG

In [0]:
def getAccountFlags(df_CycleData, v_flagType='All', v_flagSubType='All'):

    df_DIMFlagType = spark.read.table('promotion.dim_flagtype').select('FlagTypeUUID','FlagRank','FlagType','FlagSubType').filter("FlagSubType != 'FraudReport'")

    if v_flagType != 'All':
        df_DIMFlagType = df_DIMFlagType.filter(df_DIMFlagType.FlagType == v_flagType)

    if v_flagSubType != 'All':
        df_DIMFlagType = df_DIMFlagType.filter(df_DIMFlagType.FlagSubType == v_flagSubType)
        
    df_FactAccountFlag = spark.read.table('promotion.fact_accountflag').select('FlagTypeUUID','ShipToAccountUUID','FlagStartDate','FlagEndDate','PromotionUUID')
    df_AccountFlag = df_FactAccountFlag.join(df_DIMFlagType,'FlagTypeUUID','inner')


    df_CycleData_ShipTo = df_CycleData.select('ShipToAccountUUID','PromotionUUID','CycleDate').dropDuplicates()

    df_CycleData_ShipTo_Flag = (df_CycleData_ShipTo
                                .join(df_AccountFlag,['ShipToAccountUUID','PromotionUUID'],'inner')
                                .filter(df_CycleData_ShipTo.CycleDate.between(df_AccountFlag.FlagStartDate,df_AccountFlag.FlagEndDate))
                                .withColumn('RowNum',row_number().over(Window.partitionBy('ShipToAccountUUID','PromotionUUID','CycleDate').orderBy(asc('FlagRank'))))
                                .filter(col('RowNum') == 1)
                                .drop('RowNum')
                                .select('ShipToAccountUUID','PromotionUUID','CycleDate','FlagTypeUUID','FlagType','FlagSubType')
                                .drop_duplicates()
                              )
    return df_CycleData_ShipTo_Flag

In [0]:
def populateGenericComments(UpdatedBy="ADB_UpdateComments"):
    
    spark.sql(f'''
            with MultiSubscriptions as 
                (
                    select CoolSculptingID,count(Distinct iad.ConsumerSubscriptionUUID) As CNT
                    FROM promotion.fact_consumersubscription AS iad
                    WHERE iad.VersionID = 1            
                    group by all
                    Having count(Distinct iad.ConsumerSubscriptionUUID) > 1
                )
                MERGE INTO promotion.fact_invoiceaddendumdetails AS target
                USING (SELECT Distinct iad.InvoiceAddendumDetailsUUID
                       FROM promotion.fact_invoiceaddendumdetails AS iad
                       inner join MultiSubscriptions AS MS
                           ON iad.CoolSculptingID = MS.CoolSculptingID
                        inner join promotion.DIM_PatientClassification AS DP
                                ON iad.PatientClassificationUUID = DP.PatientClassificationUUID
                                AND DP.PatientClassificationName IN ("PerPatientFee")
                       Where iad.VersionID = 1 AND coalesce(iad.Comments,'' )  = '' AND iad.CreatedDate >= CURRENT_TIMESTAMP() - INTERVAL 15 MINUTES
                       ) AS source
                ON target.InvoiceAddendumDetailsUUID = source.InvoiceAddendumDetailsUUID
                WHEN MATCHED THEN 
                UPDATE SET Comments = 'ReturnSubscription',
                           UpdatedDate = CURRENT_TIMESTAMP(),
                           UpdatedBy = '{UpdatedBy}'
            ''')

    spark.sql(f'''with MAXInvoiceDate as (
                            SELECT PromotionUUID,max(InvoiceDate) AS MAXInvoiceDate 
                            FROM promotion.fact_invoiceaddendum
                            group by all
                        )
                UPDATE promotion.fact_invoiceaddendumdetails
                SET Comments = 'UpdateCycle_NewSubscription',
                    UpdatedDate = CURRENT_TIMESTAMP(),
                    UpdatedBy = '{UpdatedBy}'
                Where InvoiceAddendumDetailsUUID IN (
                    Select InvoiceAddendumDetailsUUID
                    FROM promotion.fact_invoiceaddendumdetails AS iad
                    inner join MAXInvoiceDate AS MD
                        ON cast(iad.CreatedDate AS Date) <= MD.MAXInvoiceDate 
                        AND cast(iad.UpdatedDate AS Date) > MD.MAXInvoiceDate
                        AND iad.PromotionUUID = MD.PromotionUUID
                    Where iad.VersionID = 1 AND coalesce(iad.Comments,'' ) = '' AND iad.UpdatedDate >= CURRENT_TIMESTAMP() - INTERVAL 15 MINUTES AND iad.UpdatedBy not like 'AAIOT%'
                        )''')

    spark.sql(f'''
            with MAXInvoiceDate as (
                    Select PromotionUUID,max(InvoiceDate) AS MAXInvoiceDate 
                    FROM promotion.fact_invoiceaddendum
                    group by all
                ),
            NoPatientAssociationDelayedCycle as (
                select iad.ShipToAccountUUID,iad.CycleDate,iad.PatientClassificationUUID,count(1) As CNT
                FROM promotion.fact_invoiceaddendumdetails AS iad
                inner join promotion.DIM_PatientClassification AS DP
                        ON iad.PatientClassificationUUID = DP.PatientClassificationUUID
                        AND DP.PatientClassificationName = 'NoPatientAssociationFee'
                inner join MAXInvoiceDate AS MD
                        ON iad.CreatedDate <= MD.MAXInvoiceDate
                        AND MD.PromotionUUID = iad.PromotionUUID
                WHERE iad.VersionID = 1
                group by all
            )

            UPDATE promotion.fact_invoiceaddendumdetails
            SET Comments = 'UpdateCycle_NoPatientAssociation',
                UpdatedDate = CURRENT_TIMESTAMP(),
                UpdatedBy = '{UpdatedBy}'
            Where InvoiceAddendumDetailsUUID IN (
                Select InvoiceAddendumDetailsUUID
                FROM promotion.fact_invoiceaddendumdetails AS iad
                inner join NoPatientAssociationDelayedCycle AS npd
                    ON  iad.ShipToAccountUUID = npd.ShipToAccountUUID
                    AND iad.CycleDate = npd.CycleDate
                    AND iad.PatientClassificationUUID = npd.PatientClassificationUUID
                inner join MAXInvoiceDate AS MD
                    ON iad.CreatedDate > MD.MAXInvoiceDate
                Where iad.VersionID = 1 AND (iad.Comments = '' OR coalesce(iad.Comments,'LateCycle_NoPatientAssociation' )  = 'LateCycle_NoPatientAssociation') AND iad.CreatedDate >= CURRENT_TIMESTAMP() - INTERVAL 15 MINUTES  AND iad.UpdatedBy not like 'AAIOT%'
            )
        ''')
    
    spark.sql(
        f'''
            WITH LateCycle_Comments AS (
                select iad.InvoiceAddendumDetailsUUID,
                        CASE WHEN PC.PatientClassificationName = 'PerPatientFee' THEN 'LateCycle_NewSubscription'
                            WHEN PC.PatientClassificationName = 'FollowUpVisit' THEN 'LateCycle_FollowUp'
                            WHEN PC.PatientClassificationName = 'NoPatientAssociationFee' THEN 'LateCycle_NoPatientAssociation' 
                            ELSE ''
                            END AS Comments
                From promotion.fact_invoiceaddendumdetails AS iad
                inner join promotion.dim_invoicecyclemonth AS IVC
                        ON current_timestamp() between IVC.CalendarStartTimeStamp AND IVC.CalendarEndTimeStamp
                        AND iad.CycleDate < cast(IVC.CalendarStartTimeStamp AS Date)
                        AND iad.CreatedDate > IVC.CalendarStartTimeStamp
                inner join promotion.DIM_PatientClassification AS PC
                        ON PC.PatientClassificationUUID = iad.PatientClassificationUUID
                        AND PC.Active = 1
                        AND PatientClassificationName IN ('PerPatientFee', 'FollowUpVisit','NoPatientAssociationFee')
                Where iad.VersionID = 1 AND coalesce(iad.Comments,'') = '' AND iad.CreatedDate >= CURRENT_TIMESTAMP() - INTERVAL 15 MINUTES
                )

                MERGE INTO promotion.fact_invoiceaddendumdetails AS iad
                USING LateCycle_Comments AS Late
                ON iad.InvoiceAddendumDetailsUUID = Late.InvoiceAddendumDetailsUUID AND Late.Comments <> '' AND coalesce(iad.Comments,'') = ''
                WHEN MATCHED THEN 
                UPDATE 
                SET iad.Comments = Late.Comments,
                    iad.UpdatedDate = current_timestamp(),
                    iad.UpdatedBy = '{UpdatedBy}'
        ''')
        